In [1]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

### Load and inspect the model

In [2]:
MODEL_ID = "microsoft/swin-tiny-patch4-window7-224"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

In [3]:
model.eval()

print("Device: ", device)
print("Model :", type(model).__name__)
print("Top-level modules :")
for name, module in model.named_children():
    print(name, "→", type(module).__name__)

Device:  cpu
Model : SwinForImageClassification
Top-level modules :
swin → SwinModel
classifier → Linear


**Configuration X-ray**

In [4]:
stage_dimensions = [
    model.config.embed_dim * (2 ** stage_index)
    for stage_index in range(len(model.config.depths))
]

print("Image size        :", model.config.image_size)
print("Patch size        :", model.config.patch_size)
print("Window size       :", model.config.window_size)
print("Initial dimension :", model.config.embed_dim)
print("Stage dimensions  :", stage_dimensions)
print("Stage depths      :", model.config.depths)
print("Attention heads   :", model.config.num_heads)
print("MLP ratio         :", model.config.mlp_ratio)
print("Number of classes :", model.config.num_labels)
print(
    "Absolute position embedding:",
    model.config.use_absolute_embeddings
)

Image size        : 224
Patch size        : 4
Window size       : 7
Initial dimension : 96
Stage dimensions  : [96, 192, 384, 768]
Stage depths      : [2, 2, 6, 2]
Attention heads   : [3, 6, 12, 24]
MLP ratio         : 4.0
Number of classes : 1000
Absolute position embedding: False


### Patch Embedding

**Image to patch token**

In [5]:
pixel_values = torch.randn(1, 3, 224, 224, device=device)

patch_embedding_layer = (model.swin.embeddings.patch_embeddings)

with torch.inference_mode():
    patch_tokens, grid_size = patch_embedding_layer(
        pixel_values
    )

print("Input image :", pixel_values.shape)
print("Patch grid :", grid_size)
print("Patch tokens :", patch_tokens.shape)
print("Projection Layer :", patch_embedding_layer.projection)    

Input image : torch.Size([1, 3, 224, 224])
Patch grid : (56, 56)
Patch tokens : torch.Size([1, 3136, 96])
Projection Layer : Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))


### Window-Based Self-Attention

**divide gird into 56x56 window**

In [6]:
batch_size = patch_tokens.shape[0]
height, width = grid_size
channels = patch_tokens.shape[-1]

feature_map = patch_tokens.reshape(batch_size, height, width, channels)
window_size = model.config.window_size

In [7]:
windows = (
    feature_map.reshape(
        batch_size, height // window_size,
        window_size, width // window_size,
        window_size, channels
    ).permute(0, 1, 2, 3, 4, 5).reshape(
        -1, window_size * window_size, channels
    )
)

num_of_heads = model.config.num_heads[0]
head_dimension = channels // num_of_heads

In [8]:
qkv_shape = (
    windows.shape[0], num_of_heads,
    num_of_heads, window_size * window_size,
    head_dimension
)

attention_shape = (
    windows.shape[0], num_of_heads,
    window_size * window_size,
    window_size * window_size
)

print("Feature map     :", feature_map.shape)
print("Number windows  :", windows.shape[0])
print("One window      :", windows.shape[1:])
print("Q/K/V per head  :", qkv_shape)
print("Attention matrix:", attention_shape)

Feature map     : torch.Size([1, 56, 56, 96])
Number windows  : 64
One window      : torch.Size([49, 96])
Q/K/V per head  : (64, 3, 3, 49, 32)
Attention matrix: (64, 3, 49, 49)


### Patch Merging

**Patch merging shape demonstration**

In [9]:
top_left = feature_map[:, 0::2, 0::2, :]
bottom_left = feature_map[:, 1::2, 0::2, :]
top_right = feature_map[:, 0::2, 1::2, :]
bottom_right = feature_map[:, 1::2, 1::2, :]

concatenated = torch.cat([
    top_left, bottom_left,
    top_right, bottom_right
], dim=-1)

print("Before merging :", feature_map.shape)
print("Concatenated   :", concatenated.shape)
print("After reduction: [1, 28, 28, 192]")

Before merging : torch.Size([1, 56, 56, 96])
Concatenated   : torch.Size([1, 28, 28, 384])
After reduction: [1, 28, 28, 192]


In [10]:
with torch.inference_mode():
    backbone_outputs = model.swin(
        pixel_values=pixel_values,
        return_dict=True,
    )

    classification_outputs = model(
        pixel_values=pixel_values,
        return_dict=True,
    )

print(
    "Final token representations:",
    backbone_outputs.last_hidden_state.shape,
)

print(
    "Average-pooled representation:",
    backbone_outputs.pooler_output.shape,
)

print(
    "Classification logits:",
    classification_outputs.logits.shape,
)

print("Classifier:")
print(model.classifier)

Final token representations: torch.Size([1, 49, 768])
Average-pooled representation: torch.Size([1, 768])
Classification logits: torch.Size([1, 1000])
Classifier:
Linear(in_features=768, out_features=1000, bias=True)


**Parameter count**

In [11]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

backbone_parameters = sum(
    parameter.numel()
    for parameter in model.swin.parameters()
)

classifier_parameters = sum(
    parameter.numel()
    for parameter in model.classifier.parameters()
)

print(f"Total parameters     : {total_parameters:,}")
print(f"Backbone parameters  : {backbone_parameters:,}")
print(f"Classifier parameters: {classifier_parameters:,}")

Total parameters     : 28,288,354
Backbone parameters  : 27,519,354
Classifier parameters: 769,000
